In [ ]:
import json
from pathlib import Path

from pyspark.sql import SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("InsuranceLakehouseBronzeIngestion")
    .master("local[*]")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")

In [ ]:
current_path = Path.cwd()

print(f"Current working directory: {current_path}")

In [ ]:
project_root = current_path.parent

config_path = project_root / "config" / "bronze_sources.json"
source_path = project_root / "data" / "source"

print(f"Project root: {project_root}")
print(f"Config path:  {config_path}")
print(f"Source path:  {source_path}")

In [ ]:
with open(config_path, "r", encoding="utf-8") as config_file:
    config = json.load(config_file)

sources = config["sources"]

print(f"Configured sources: {len(sources)}")

for source in sources:
    print(f"- {source['source_name']} | active: {source['active']}")

In [ ]:
for source in sources:
    if source["active"]:
        file_path = source_path / source["file_name"]

        if file_path.exists():
            print(f"OK      | {source['source_name']} | {source['file_name']}")
        else:
            print(f"MISSING | {source['source_name']} | {source['file_name']}")

In [ ]:
for source in sources:
    if not source["active"]:
        continue

    file_path = source_path / source["file_name"]

    df = (
        spark.read
        .format(source["file_format"])
        .option("header", source["header"])
        .option("delimiter", source["delimiter"])
        .load(str(file_path))
    )

    print(f"Source: {source['source_name']}")
    df.show()